In [3]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import linear_sum_assignment
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [5]:
PB = [f"PB{i}" for i in range(1, 7)]
INT = [f"INT{i}" for i in range(1, 5)]
ORG = [f"ORG{i}" for i in range(1, 3)]
TMS = [f"TMS{i}" for i in range(1, 4)]
CP = [f"CP{i}" for i in range(1, 4)]
EFA_ITEMS = PB + INT + ORG + TMS + CP

COLUMN_RENAME = {
    "PB1_OperationalEfficiency": "PB1",
    "PB2_CustomerService": "PB2",
    "PB3_CompetitiveAdvantage": "PB3",
    "PB4_CostReduction": "PB4",
    "PB5_Innovation": "PB5",
    "PB6_NetBenefitsExceedRisks": "PB6",
    "INT1_PlannedInvestment12Months": "INT1",
    "INT2_BudgetAllocated": "INT2",
    "INT3_VendorEngagement": "INT3",
    "INT4_StrategicCommitment": "INT4",
    "ORG1_SkilledPersonnel": "ORG1",
    "ORG2_InfrastructureAndData": "ORG2",
    "TMS1_LeadershipSupport": "TMS1",
    "TMS2_ResourceProvision": "TMS2",
    "TMS3_StrategicVision": "TMS3",
    "CP1_CompetitorPressure": "CP1",
    "CP2_ClientExpectations": "CP2",
    "CP3_IndustryTrendAndPolicy": "CP3",
    "C1_AdoptionStage": "C1",
    "A1_Role": "A1",
    "A2_InstitutionType": "A2",
    "A3_FirmSize": "A3",
    "A4_Region": "A4",
}

ITEM_STATEMENTS = {
    "INT1": "Our institution plans to allocate financial resources to GenAI initiatives within the next 12 months.",
    "INT2": "Our institution has already budgeted or ring-fenced funds for GenAI-related activities.",
    "INT3": "Our institution is actively evaluating or engaging with vendors or implementation partners for GenAI.",
    "INT4": "Our institution is committed to integrating GenAI into its medium- to long-term strategic plans.",
    "PB1": "GenAI can significantly improve our institution's operational efficiency (e.g. back-office automation and compliance drafting).",
    "PB2": "GenAI can enhance the quality of customer service and client engagement (e.g. chatbots and personalised advice).",
    "PB3": "Adopting GenAI would provide our institution with a meaningful competitive advantage over non-adopting banks.",
    "PB4": "GenAI can help our institution reduce operational costs over the medium to long term.",
    "PB5": "GenAI can support innovation and the development of new banking products or services.",
    "PB6": "Overall, the anticipated benefits of GenAI outweigh the potential risks and implementation costs.",
    "ORG1": "Our institution has the skilled personnel and expertise required to implement and manage GenAI systems.",
    "ORG2": "We have adequate data assets, technology infrastructure, and digital tools to support GenAI projects.",
    "TMS1": "The senior leadership of our institution actively supports the adoption and integration of Generative AI.",
    "TMS2": "Senior leadership provides the necessary budget, time, and human resources for GenAI initiatives.",
    "TMS3": "Senior leadership communicates a clear strategic vision for how GenAI will create value for our institution.",
    "CP1": "Other banks in Ghana are adopting GenAI, creating competitive pressure for our institution to follow.",
    "CP2": "Our clients increasingly expect AI-enabled banking services such as intelligent chatbots and personalised insights.",
    "CP3": "Industry trends and Ghana's National AI Strategy indicate that GenAI adoption is becoming a strategic necessity.",
}

CONSTRUCT_OF = {**{x: "Perceived Benefits" for x in PB},
                **{x: "Investment Intention" for x in INT},
                **{x: "Organisational Readiness" for x in ORG},
                **{x: "Top Management Support" for x in TMS},
                **{x: "Competitive Pressure" for x in CP}}


In [6]:
def cronbach_alpha(frame: pd.DataFrame) -> float:
    x = frame.dropna().astype(float)
    k = x.shape[1]
    item_variance = x.var(axis=0, ddof=1).sum()
    total_variance = x.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_variance / total_variance)


def item_descriptives(df: pd.DataFrame, items: list[str]) -> pd.DataFrame:
    rows = []
    for item in items:
        x = df[item].dropna().astype(float)
        rows.append({
            "Item": item,
            "Statement": ITEM_STATEMENTS[item],
            "Mean": x.mean(),
            "SD": x.std(ddof=1),
            "Skew": stats.skew(x, bias=False),
            "Kurt.": stats.kurtosis(x, fisher=True, bias=False),
            "N": x.size,
        })
    return pd.DataFrame(rows)


def kmo_bartlett(x: pd.DataFrame) -> tuple[float, float, int, float]:
    """Return overall KMO, Bartlett chi-square, df and p-value."""
    z = x.dropna().astype(float)
    r = z.corr().to_numpy()
    inv_r = np.linalg.pinv(r)
    d = np.sqrt(np.diag(inv_r))
    partial = -inv_r / np.outer(d, d)
    np.fill_diagonal(partial, 0.0)
    r0 = r.copy()
    np.fill_diagonal(r0, 0.0)
    kmo = np.sum(r0 ** 2) / (np.sum(r0 ** 2) + np.sum(partial ** 2))

    n, p = z.shape
    det_r = max(np.linalg.det(r), np.finfo(float).tiny)
    chi2 = -(n - 1 - (2 * p + 5) / 6) * np.log(det_r)
    dof = p * (p - 1) // 2
    p_value = stats.chi2.sf(chi2, dof)
    return kmo, chi2, dof, p_value


def varimax(loadings: np.ndarray, gamma: float = 1.0,
            max_iter: int = 1000, tol: float = 1e-8) -> np.ndarray:
    """Orthogonal varimax rotation."""
    p, k = loadings.shape
    rotation = np.eye(k)
    previous = 0.0
    for _ in range(max_iter):
        transformed = loadings @ rotation
        u, s, vh = np.linalg.svd(
            loadings.T @ (transformed ** 3 - (gamma / p) * transformed @
                          np.diag(np.diag(transformed.T @ transformed)))
        )
        rotation = u @ vh
        current = s.sum()
        if previous and current / previous < 1 + tol:
            break
        previous = current
    return loadings @ rotation


def principal_axis_factoring(corr: np.ndarray, n_factors: int = 5,
                             max_iter: int = 500, tol: float = 1e-8) -> tuple[np.ndarray, np.ndarray]:
    """
    Principal axis factoring using iterative communalities followed by varimax.
    Returns rotated loadings and eigenvalues of the original correlation matrix.
    """
    p = corr.shape[0]
    inv_corr = np.linalg.pinv(corr)
    communalities = np.clip(1 - 1 / np.diag(inv_corr), 0.01, 0.99)

    loadings = None
    for _ in range(max_iter):
        reduced = corr.copy()
        np.fill_diagonal(reduced, communalities)
        eigvals, eigvecs = np.linalg.eigh(reduced)
        order = np.argsort(eigvals)[::-1]
        eigvals, eigvecs = eigvals[order], eigvecs[:, order]
        positive = np.clip(eigvals[:n_factors], 0, None)
        loadings = eigvecs[:, :n_factors] * np.sqrt(positive)
        new_communalities = np.sum(loadings ** 2, axis=1)
        if np.max(np.abs(new_communalities - communalities)) < tol:
            break
        communalities = np.clip(new_communalities, 0.0, 0.999999)

    rotated = varimax(loadings)
    original_eigenvalues = np.linalg.eigvalsh(corr)[::-1]
    return rotated, original_eigenvalues


def align_factors(loadings: np.ndarray) -> np.ndarray:
    """
    Reorder the arbitrary EFA factors to the chapter's reporting convention:
    F1 = ORG/CP, F2 = PB, F3 = INT/CP, F4 = TMS/INT, F5 = residual.
    """
    index = {item: i for i, item in enumerate(EFA_ITEMS)}
    target_groups = [ORG, PB, INT, TMS]
    scores = np.array([
        [np.sum(np.abs(loadings[[index[x] for x in group], f])) for f in range(loadings.shape[1])]
        for group in target_groups
    ])
    rows, cols = linear_sum_assignment(-scores)
    assigned = {row: col for row, col in zip(rows, cols)}
    order = [assigned[i] for i in range(4)]
    order.append(next(f for f in range(loadings.shape[1]) if f not in order))
    aligned = loadings[:, order]

    # Factor signs have no substantive meaning; orient main construct loadings positively.
    for f, group in enumerate(target_groups):
        idx = [index[x] for x in group]
        if aligned[idx, f].sum() < 0:
            aligned[:, f] *= -1
    return aligned


def vif_values(x: pd.DataFrame) -> pd.Series:
    # VIF must be calculated with an intercept in the auxiliary regressions.
    numeric = sm.add_constant(x.astype(float), has_constant="add")
    values = pd.Series(
        [variance_inflation_factor(numeric.to_numpy(), i) for i in range(numeric.shape[1])],
        index=numeric.columns,
    )
    return values.drop("const")


def regression_table(model, predictors: list[str], labels: dict[str, str],
                     vif: pd.Series | None = None, include_intercept: bool = True) -> pd.DataFrame:
    rows = []
    if include_intercept:
        rows.append({
            "Predictor": "Intercept",
            "Beta (b)": model.params["const"],
            "Std. Error": model.bse["const"],
            "t-statistic": model.tvalues["const"],
            "p-value": model.pvalues["const"],
            "VIF": np.nan,
        })
    for variable in predictors:
        rows.append({
            "Predictor": labels.get(variable, variable),
            "Beta (b)": model.params[variable],
            "Std. Error": model.bse[variable],
            "t-statistic": model.tvalues[variable],
            "p-value": model.pvalues[variable],
            "VIF": np.nan if vif is None else vif[variable],
        })
    out = pd.DataFrame(rows)
    out.loc[len(out)] = ["R-squared", model.rsquared, np.nan, np.nan, np.nan, np.nan]
    if include_intercept:
        out.loc[len(out)] = ["Adjusted R-squared", model.rsquared_adj, np.nan, np.nan, np.nan, np.nan]
    return out


In [ ]:
def build_tables_and_figure(df: pd.DataFrame, output_dir: Path) -> dict[str, pd.DataFrame]:
    output_dir.mkdir(parents=True, exist_ok=True)
    n = len(df)
    tables: dict[str, pd.DataFrame] = {}

    # Table 4: Demographic profile
    role = df["A1"].replace({"IT intern": "IT Intern"})
    role_group = role.where(role.isin([
        "Senior Functional Manager", "CEO or Managing Director",
        "Innovation or Digital Transformation Manager", "CIO or CTO"
    ]), "Other roles (Head of Operations, Risk Manager, Business Transformation Manager, etc.)")
    role_group = role_group.replace({"CIO or CTO": "Chief Information Officer or Chief Technology Officer"})

    profile_specs = [
        ("Respondent Role (A1)", role_group, [
            "Senior Functional Manager", "CEO or Managing Director",
            "Innovation or Digital Transformation Manager",
            "Chief Information Officer or Chief Technology Officer",
            "Other roles (Head of Operations, Risk Manager, Business Transformation Manager, etc.)"]),
        ("Institution Type (A2)", df["A2"].replace({"Commercial bank": "Commercial bank (universal banking licence)"}), [
            "Commercial bank (universal banking licence)", "Savings and loans company",
            "Rural or community bank", "Microfinance institution"]),
        ("Firm Size (A3)", df["A3"].map({1: "Fewer than 50 employees", 2: "50 to 249 employees",
                                         3: "250 to 999 employees", 4: "1,000 or more employees"}), [
            "Fewer than 50 employees", "50 to 249 employees", "250 to 999 employees", "1,000 or more employees"]),
        ("Headquarters Region (A4)", df["A4"].replace({
            "Ashanti": "Ashanti Region", "Central": "Central Region", "Western": "Western Region",
            "Northern": "Northern Region"}), [
            "Greater Accra", "Ashanti Region", "Central Region", "Western Region", "Northern Region"]),
    ]
    demographic_rows = []
    for variable, series, categories in profile_specs:
        counts = series.value_counts()
        for category in categories:
            frequency = int(counts.get(category, 0))
            demographic_rows.append({"Profile Variable": variable, "Category": category,
                                     "Frequency": frequency, "Percentage (%)": 100 * frequency / n})
        demographic_rows.append({"Profile Variable": "", "Category": "Total", "Frequency": n, "Percentage (%)": 100.0})
    tables["Table_04_Demographic_Profile"] = pd.DataFrame(demographic_rows)

    # Table 5: Adoption stage distribution, including a zero-frequency Stage 1
    stage_labels = {
        1: "Stage 1: Not considering", 2: "Stage 2: Exploring or researching",
        3: "Stage 3: Planning or budgeting", 4: "Stage 4: Piloting or testing",
        5: "Stage 5: Implemented in selected areas", 6: "Stage 6: Fully implemented",
    }
    counts = df["C1"].value_counts().reindex(range(1, 7), fill_value=0)
    pct = counts / n * 100
    t5 = pd.DataFrame({"Adoption Stage": [stage_labels[i] for i in range(1, 7)],
                       "Frequency": counts.to_numpy(), "Percentage (%)": pct.to_numpy(),
                       "Cumulative %": pct.cumsum().to_numpy()})
    t5.loc[len(t5)] = ["Total", n, 100.0, np.nan]
    tables["Table_05_Adoption_Stage_Distribution"] = t5

    # Tables 6 to 10: Item-level descriptive statistics
    tables["Table_06_Investment_Intention_Descriptives"] = item_descriptives(df, INT)
    tables["Table_07_Perceived_Benefits_Descriptives"] = item_descriptives(df, PB)
    tables["Table_08_Organisational_Readiness_Descriptives"] = item_descriptives(df, ORG)
    tables["Table_09_Top_Management_Support_Descriptives"] = item_descriptives(df, TMS)
    tables["Table_10_Competitive_Pressure_Descriptives"] = item_descriptives(df, CP)

    # EFA diagnostics, pattern matrix and variance explained
    efa_data = df[EFA_ITEMS].dropna().astype(float)
    corr = efa_data.corr().to_numpy()
    kmo, bartlett_chi2, bartlett_df, bartlett_p = kmo_bartlett(efa_data)
    raw_loadings, eigenvalues = principal_axis_factoring(corr, n_factors=5)
    loadings = align_factors(raw_loadings)
    communalities = np.sum(loadings ** 2, axis=1)

    efa_rows = []
    for i, item in enumerate(EFA_ITEMS):
        row = {"Item": item, "Construct": CONSTRUCT_OF[item]}
        for f in range(5):
            value = loadings[i, f]
            row[f"F{f + 1}"] = value if abs(value) >= 0.30 else np.nan
        row["h2"] = communalities[i]
        efa_rows.append(row)
    tables["Table_11_EFA_Pattern_Matrix"] = pd.DataFrame(efa_rows)

    ss = np.sum(loadings ** 2, axis=0)
    prop = ss / len(EFA_ITEMS)
    dominant = ["Organisational Readiness/CP", "Perceived Benefits", "Investment Intention/CP",
                "Top Management Support/INT", "Residual"]
    t12 = pd.DataFrame({"Factor": [f"F{i}" for i in range(1, 6)], "SS Loadings": ss,
                        "Proportion of Variance": prop, "Cumulative Variance": np.cumsum(prop),
                        "Dominant Construct": dominant})
    t12.loc[len(t12)] = ["Total", ss.sum(), prop.sum(), np.nan, ""]
    tables["Table_12_Variance_Explained"] = t12

    # Composite scores for Tables 13 to 16
    constructs = {"PB": PB, "INT": INT, "ORG": ORG, "TMS": TMS, "CP": CP}
    for name, items in constructs.items():
        df[name] = df[items].mean(axis=1)

    construct_names = {
        "PB": "Perceived Benefits (PB)", "INT": "Investment Intention (INT)",
        "ORG": "Organisational Readiness (ORG)", "TMS": "Top Management Support (TMS)",
        "CP": "Competitive and Environmental Pressure (CP)"}
    reliability_rows = []
    for key, items in constructs.items():
        alpha = cronbach_alpha(df[items])
        assessment = "Good" if alpha >= 0.80 else "Acceptable" if alpha >= 0.70 else "Below threshold"
        reliability_rows.append({"Construct": construct_names[key], "Items": len(items),
                                 "Mean": df[key].mean(), "SD": df[key].std(ddof=1),
                                 "Alpha": alpha, "Assessment": assessment})
    tables["Table_13_Construct_Descriptives_Reliability"] = pd.DataFrame(reliability_rows)

    # Pearson r with significance stars in a lower-triangular presentation
    keys = ["PB", "INT", "ORG", "TMS", "CP"]
    corr_rows = []
    for i, row_key in enumerate(keys):
        row = {"Construct": construct_names[row_key]}
        for j, col_key in enumerate(keys):
            if j > i:
                row[col_key] = ""
            elif i == j:
                row[col_key] = "1.000"
            else:
                r, p = stats.pearsonr(df[row_key], df[col_key])
                stars = "**" if p < 0.01 else "*" if p < 0.05 else ""
                row[col_key] = f"{r:.3f}{stars}"
        corr_rows.append(row)
    tables["Table_14_Pearson_Correlation_Matrix"] = pd.DataFrame(corr_rows)

    # Table 15: H1 hierarchical/full regression model
    predictors15 = ["PB", "ORG", "TMS", "CP"]
    x15 = df[predictors15]
    model15 = sm.OLS(df["INT"], sm.add_constant(x15)).fit()
    vif15 = vif_values(x15)
    labels15 = {"PB": "Perceived Benefits (H1 focal predictor)",
                "ORG": "Organisational Readiness (control)",
                "TMS": "Top Management Support (control)",
                "CP": "Competitive Pressure (control)"}
    tables["Table_15_H1_Hierarchical_Regression"] = regression_table(
        model15, predictors15, labels15, vif15, include_intercept=True)

    # Table 16: Mean-centred moderation model
    df["PB_c"] = df["PB"] - df["PB"].mean()
    df["FirmSize_c"] = df["A3"] - df["A3"].mean()
    df["PB_x_FirmSize"] = df["PB_c"] * df["FirmSize_c"]
    predictors16 = ["PB_c", "FirmSize_c", "PB_x_FirmSize", "ORG", "TMS", "CP"]
    x16 = df[predictors16]
    model16 = sm.OLS(df["INT"], sm.add_constant(x16)).fit()
    vif16 = vif_values(x16)
    labels16 = {"PB_c": "Perceived Benefits: centred (H2 focal IV)",
                "FirmSize_c": "Firm Size: centred (moderator)",
                "PB_x_FirmSize": "PB x Firm Size: interaction term (H2)",
                "ORG": "Organisational Readiness (control)",
                "TMS": "Top Management Support (control)",
                "CP": "Competitive Pressure (control)"}
    tables["Table_16_H2_Moderation_Analysis"] = regression_table(
        model16, predictors16, labels16, vif16, include_intercept=False)

    # Figure 5: scree plot and absolute factor-loading heatmap
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={"width_ratios": [0.9, 1.7]})

    ax = axes[0]
    factor_numbers = np.arange(1, 9)  # Chapter figure displays the first eight eigenvalues
    ax.plot(factor_numbers, eigenvalues[:8], marker="o", linewidth=1.7, color="#183a66")
    ax.axhline(1.0, color="#d62728", linestyle="--", linewidth=1.2, label="Eigenvalue = 1 criterion")
    ax.axvline(5.5, color="#bdbdbd", linestyle=":", linewidth=1.0, label="Retained factors (n=5)")
    for x, y in zip(factor_numbers[:5], eigenvalues[:5]):
        ax.annotate(f"{y:.3f}", (x, y), xytext=(4, 5), textcoords="offset points", fontsize=8)
    ax.text(0.03, 0.04, f"KMO = {kmo:.3f}\nBartlett p < 0.001\nN = {len(efa_data)}",
            transform=ax.transAxes, fontsize=9,
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85, "edgecolor": "#bbbbbb"})
    ax.set_title("Figure 4.1: Scree Plot")
    ax.set_xlabel("Factor Number")
    ax.set_ylabel("Eigenvalue")
    ax.set_xticks(factor_numbers)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8, loc="upper right")

    ax = axes[1]
    abs_loadings = np.abs(loadings)
    image = ax.imshow(abs_loadings, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_title("Figure 4.2: Factor Loading Matrix\n(Absolute loadings; values < 0.30 suppressed)")
    ax.set_xticks(range(5), ["F1\n(ORG/CP)", "F2\n(PB)", "F3\n(INT/CP)", "F4\n(TMS/INT)", "F5"])
    ax.set_yticks(range(len(EFA_ITEMS)), EFA_ITEMS)
    for i in range(abs_loadings.shape[0]):
        for j in range(abs_loadings.shape[1]):
            value = loadings[i, j]
            if abs(value) >= 0.30:
                colour = "white" if abs(value) >= 0.68 else "#263238"
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7, color=colour,
                        fontweight="bold" if abs(value) >= 0.50 else "normal")
    cbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Absolute Factor Loading")
    ax.set_xlabel("Rotated Factor")
    ax.set_ylabel("Survey Item")

    fig.suptitle("Figure 5: Scree Plot and Factor Loading Matrix", fontsize=14, fontweight="bold")
    fig.text(0.5, 0.01,
             f"Source: Field Survey (2026). Extraction: Principal Axis Factoring; Rotation: Varimax. "
             f"KMO={kmo:.3f}; Bartlett chi2={bartlett_chi2:.3f}, df={bartlett_df}, p<0.001.",
             ha="center", fontsize=8)
    fig.tight_layout(rect=[0, 0.04, 1, 0.95])
    fig.savefig(output_dir / "Figure_05_Scree_Plot_and_Factor_Loading_Matrix.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Save each table to CSV and all tables to one Excel workbook.
    for name, table in tables.items():
        table.to_csv(output_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(output_dir / "all_tables.xlsx", engine="openpyxl") as writer:
        for name, table in tables.items():
            sheet_name = name.replace("Table_", "T")[:31]
            table.to_excel(writer, sheet_name=sheet_name, index=False)

    # A compact diagnostics file makes the EFA assumptions auditable.
    pd.DataFrame([{
        "N": len(efa_data), "KMO": kmo, "Bartlett chi-square": bartlett_chi2,
        "Bartlett df": bartlett_df, "Bartlett p-value": bartlett_p,
        "Number of retained factors": 5,
    }]).to_csv(output_dir / "EFA_diagnostics.csv", index=False)

    return tables


In [9]:
# File locations
INPUT_FILE = Path(r"C:\Users\LW344XJ\Downloads\Chapter 4\Final_Data.csv")
OUTPUT_DIR = Path("chapter4_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load and prepare the survey dataset
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig").rename(columns=COLUMN_RENAME)

required = {"A1", "A2", "A3", "A4", "C1", *EFA_ITEMS}
missing = sorted(required.difference(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

numeric_columns = ["A3", "C1", *EFA_ITEMS]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="raise")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


Rows: 116
Columns: 29


,ResponseID,A1,A2,A3,A4,PB1,PB2,PB3,PB4,PB5,PB6,C1,INT1,INT2,INT3,INT4,D1_DigitalMaturity,D2_GenAIFamiliarity,ORG1,ORG2,TMS1,TMS2,TMS3,CP1,CP2,CP3,G1_BiggestMotivator,G2_ChallengesAndConcerns,G3_OtherObservations
0,1,Other,Commercial bank,4,Greater Accra,5,5,5,5,5,3,5,4,4,4,4,4,3,4,4,4,4,3,4,4,4,Improving efficiency,Privacy concerns,NaN
1,2,Senior Functional Manager,Commercial bank,2,Greater Accra,4,5,5,3,4,4,6,5,5,3,5,4,5,5,5,4,4,5,4,4,5,Improving efficiency and reducing costs throug...,Data privacy risks# accuracy of outputs# high ...,NaN
2,3,Innovation or Digital Transformation Manager,Commercial bank,4,Greater Accra,5,4,5,4,5,5,6,5,5,5,5,4,4,4,5,4,4,5,5,5,5,NaN,NaN,NaN
3,4,Senior Functional Manager,Commercial bank,3,Greater Accra,4,4,4,4,4,4,3,4,4,4,4,3,3,4,4,4,4,4,4,4,4,NaN,NaN,NaN
4,5,Senior Functional Manager,Commercial bank,4,Greater Accra,4,4,4,4,4,4,2,3,3,3,4,4,3,4,4,4,4,3,4,4,4,NaN,NaN,NaN


In [ ]:
# Generate all tables and figures
tables = build_tables_and_figure(df.copy(), OUTPUT_DIR)

print(f"Generated {len(tables)} tables.")
print(f"Outputs saved to: {OUTPUT_DIR.resolve()}")


Generated 13 tables.
Outputs saved to: C:\Users\LW344XJ\Downloads\Chapter 4\chapter4_outputs


In [ ]:
# Preview the generated table names
for table_name in tables:
    print(table_name)
